# Deep Neural Decision Tree (DNDT) and Forest (DNDF) Reliability Study
### Universal Kaggle & Google Colab GPU Runner for COVID-RARS

**Reference Paper:**  
Rofiqul Islam, Nihad Karim Chowdhury, and Muhammad Ashad Kabir. *"Robust COVID-19 detection from cough sounds using deep neural decision tree and forest: A comprehensive cross-datasets evaluation."* Expert Systems with Applications, Vol. 310, 2026, 131235. [DOI: 10.1016/j.eswa.2026.131235](https://doi.org/10.1016/j.eswa.2026.131235)

**Scientific Objective:**  
Evaluate whether differentiable DNDT/DNDF models sustain strong internal discrimination on Coswara respiratory audio across cough, speech, and breath, and test their stability under chronological drift and external transfer to COUGHVID.

## 1. Environment Setup & GPU Check (Works on Kaggle & Colab)

In [ ]:
# Install dependencies
%pip install -q torch scikit-learn imbalanced-learn pandas numpy matplotlib seaborn

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch Version: {torch.__version__}")
print(f"Execution Device: {device}")
if device == "cuda":
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## 2. Auto-Detect Environment (Kaggle vs. Colab vs. Local) & Clone Repo

In [ ]:
import os
import sys
import glob

# 1. Check if running on Kaggle
IS_KAGGLE = Path('/kaggle').exists()
IS_COLAB = 'google.colab' in sys.modules or Path('/content').exists()

if IS_KAGGLE:
    print("Detected Environment: KAGGLE")
    work_dir = Path('/kaggle/working')
    repo_root = work_dir / 'Covid-RARS'
    if not (repo_root / 'src').exists():
        !git clone https://github.com/ishaaaan17/Covid-RARS.git /kaggle/working/Covid-RARS
    output_dir = work_dir / 'reports/dndf'
elif IS_COLAB:
    print("Detected Environment: GOOGLE COLAB")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception:
        pass
    work_dir = Path('/content')
    repo_root = Path('/content/drive/MyDrive/Covid-RARS')
    if not (repo_root / 'src').exists():
        repo_root = work_dir / 'Covid-RARS'
        if not (repo_root / 'src').exists():
            !git clone https://github.com/ishaaaan17/Covid-RARS.git /content/Covid-RARS
    output_dir = Path('/content/drive/MyDrive/Covid-RARS/reports/dndf') if Path('/content/drive/MyDrive').exists() else (work_dir / 'reports/dndf')
else:
    print("Detected Environment: LOCAL")
    repo_root = Path('.').resolve()
    output_dir = repo_root / 'reports/dndf'

output_dir.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(repo_root / 'src'))
print(f"Active Repository Root: {repo_root}")
print(f"Output Directory: {output_dir}")

## 3. Load Engineered Feature Banks (Auto-Search in Kaggle/Colab/Drive)

In [ ]:
from covid_rars.dndf_models import DNDFClassifier, NeuralDecisionTree, NeuralDecisionForest
from covid_rars.dndf_stages import run_dndf_reliability_pipeline
from covid_rars.dndf_reporting import build_dndf_summary_table

# Search paths dynamically across Kaggle input, Colab Drive, and local folders
candidate_feature_paths = [
    *list(Path('/kaggle/input').glob('**/features_compare_is10_merged.csv')),
    *list(Path('/kaggle/input').glob('**/features_compare_is10_top800.csv')),
    *list(Path('/content').glob('**/features_compare_is10_merged.csv')),
    *list(Path('/content').glob('**/features_compare_is10_top800.csv')),
    Path('/content/drive/MyDrive/processed/features_compare_is10_merged.csv'),
    Path('/content/drive/MyDrive/Covid-RARS/data/processed/features_compare_is10_merged.csv'),
    repo_root / "data/processed/features_compare_is10_merged.csv",
    repo_root / "data/processed/features_compare_is10_top800.csv",
    Path('data/processed/features_compare_is10_merged.csv'),
]

features_path = None
for p in candidate_feature_paths:
    if p.exists():
        features_path = p
        break

candidate_external_paths = [
    *list(Path('/kaggle/input').glob('**/coughvid_metadata_compare_is10_external.csv')),
    *list(Path('/kaggle/input').glob('**/features_compare_is10_coughvid_cough_top800.csv')),
    Path('/content/drive/MyDrive/processed/coughvid_metadata_compare_is10_external.csv'),
    repo_root / "data/processed/features_compare_is10_coughvid_cough_top800.csv",
]

external_path = None
for p in candidate_external_paths:
    if p.exists():
        external_path = p
        break

if features_path is not None:
    print(f"Found features matrix at: {features_path}")
    features_df = pd.read_csv(features_path)
    feat_cols = [c for c in features_df.columns if c not in ["participant_id", "recording_id", "modality", "split", "label_binary", "date"]]
    if len(feat_cols) > 800:
        print(f"Detected full feature matrix with {len(feat_cols)} columns. Subsampling top 800 for efficient training.")
        selected_cols = ["participant_id", "recording_id", "modality", "split", "label_binary"] + feat_cols[:800]
        selected_cols = [c for c in selected_cols if c in features_df.columns]
        features_df = features_df[selected_cols].copy()
else:
    print("Notice: Features CSV not found. Generating standardized benchmark frame for demonstration.")
    n_samples = 400
    n_feats = 100
    rng = np.random.RandomState(42)
    records = []
    for i in range(n_samples):
        p_id = f"p_{i:04d}"
        lbl = "positive" if rng.rand() > 0.65 else "negative"
        split = "train" if i < 280 else ("val" if i < 340 else "test")
        for mod in ["cough", "breath", "speech"]:
            row = {"participant_id": p_id, "recording_id": f"{p_id}_{mod}", "label_binary": lbl, "modality": mod, "split": split}
            vec = rng.randn(n_feats) + (1.0 if lbl == "positive" else -0.5)
            for f_idx in range(n_feats):
                row[f"feat_{f_idx:03d}"] = vec[f_idx]
            records.append(row)
    features_df = pd.DataFrame(records)

external_df = pd.read_csv(external_path) if (external_path and external_path.exists()) else None
print(f"Source Features Matrix Shape: {features_df.shape}")
if external_df is not None:
    print(f"External Features Matrix Shape: {external_df.shape}")

## 4. Run End-to-End DNDT & DNDF Reliability Pipeline

In [ ]:
artifacts = run_dndf_reliability_pipeline(
    features_df=features_df,
    external_features_df=external_df,
    modalities=["cough", "breath", "speech"],
    seeds=[1, 2, 5, 12, 40],
    num_trees=20,
    depth=4,
    used_features_rate=0.8,
    learning_rate=0.01,
    max_epochs=40,
    patience=8,
    use_smote=True,
    device=device,
    output_dir=output_dir,
)

print("\n=== DNDT / DNDF FINAL SUMMARY TABLE ===")
display(artifacts.final_summary_table)

## 5. Visualizing Calibration and Reliability Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Summary Bar Chart of Track A vs Track B vs Track C
summary = artifacts.final_summary_table
if not summary.empty:
    axes[0].barh(summary["track"] + " : " + summary["modality"] + " (" + summary["model_name"] + ")", summary["mean_auroc"], color="teal", alpha=0.8)
    axes[0].set_xlim(0.4, 1.0)
    axes[0].set_xlabel("AUROC")
    axes[0].set_title("DNDT / DNDF Validation Ladder Performance")
    axes[0].grid(axis="x", linestyle="--", alpha=0.6)

# 2. Decision Curve Net Benefit
dca = artifacts.dca_summary
if not dca.empty:
    for key, grp in dca.groupby("model_name"):
        axes[1].plot(grp["threshold_probability"], grp["net_benefit_model"], label=f"{key}", lw=2)
    first_grp = list(dca.groupby("model_name"))[0][1]
    axes[1].plot(first_grp["threshold_probability"], first_grp["net_benefit_all"], label="Treat All", linestyle=":", color="gray")
    axes[1].axhline(0, color="black", linestyle="--", label="Treat None")
    axes[1].set_xlabel("Threshold Probability")
    axes[1].set_ylabel("Net Benefit")
    axes[1].set_title("Decision Curve Analysis (DCA)")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

## 6. Exporting Results & Publication Artifacts

In [ ]:
print("All DNDT / DNDF artifacts and tables generated successfully:")
print(f" - {output_dir}/dndf_final_validation_summary.csv")
print(f" - {output_dir}/dndf_calibration_summary.csv")
print(f" - {output_dir}/dndf_operating_points.csv")
print(f" - {output_dir}/dndf_decision_curves.csv")
print(f" - {output_dir}/dndf_bootstrap_ci.csv")